# Notebook 07 — NOC Regions Integration
## LA 2028 Olympic Games Strategic Playbook | SportsFanatics Consulting Agency

**Purpose:** Integrate `noc_regions.csv` into all downstream outputs to replace raw 3-letter NOC codes with human-readable country/region names. Adds continent groupings for geographic aggregation.

**Outputs:**
- `data/processed/noc_regions_clean.csv` — cleaned NOC→region lookup with continent column
- Enriched versions of all key output tables with `region` and `continent` columns
- 4 new continental aggregate charts
- 2 new enriched summary tables

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE       = Path('/Volumes/D Drive/Data analysis/Olympic data analysis')
RAW        = BASE / 'data' / 'raw'
PROCESSED  = BASE / 'data' / 'processed'
TABLES     = BASE / 'outputs' / 'tables'
FIGURES    = BASE / 'outputs' / 'figures'
PROCESSED.mkdir(parents=True, exist_ok=True)

# ── Color palette ──────────────────────────────────────────────────────────
OLYMPIC_BLUE   = '#0085C7'
OLYMPIC_YELLOW = '#F4C300'
OLYMPIC_GREEN  = '#009F6B'
OLYMPIC_RED    = '#DF0024'
OLYMPIC_BLACK  = '#000000'
BG_COLOR       = '#F8F9FA'

CONTINENT_COLORS = {
    'Europe':   OLYMPIC_BLUE,
    'Americas': OLYMPIC_RED,
    'Asia':     OLYMPIC_YELLOW,
    'Africa':   OLYMPIC_GREEN,
    'Oceania':  OLYMPIC_BLACK,
    'Other':    '#AAAAAA',
}

PLOTLY_LAYOUT = dict(
    template='plotly_white',
    font_family='Arial',
    font_size=12,
    title_font_size=16,
    margin=dict(l=60, r=40, t=70, b=60),
)

print('Libraries loaded.')

Libraries loaded.


## 1. Load & Clean NOC Regions

In [2]:
# ── Load raw noc_regions ───────────────────────────────────────────────────
noc_raw = pd.read_csv(RAW / 'noc_regions.csv')
print(f'Shape: {noc_raw.shape}')
print(noc_raw.head(10))
print(f'\nNull regions: {noc_raw["region"].isna().sum()}')
print(f'Unique regions: {noc_raw["region"].nunique()}')

Shape: (230, 3)
   NOC       region                 notes
0  AFG  Afghanistan                   NaN
1  AHO      Curacao  Netherlands Antilles
2  ALB      Albania                   NaN
3  ALG      Algeria                   NaN
4  AND      Andorra                   NaN
5  ANG       Angola                   NaN
6  ANT      Antigua   Antigua and Barbuda
7  ANZ    Australia           Australasia
8  ARG    Argentina                   NaN
9  ARM      Armenia                   NaN

Null regions: 3
Unique regions: 206


In [3]:
# ── Build continent mapping ────────────────────────────────────────────────
# Map NOC codes to continents for geographic aggregation
# Based on IOC continental associations

CONTINENT_MAP = {
    # Europe
    'ALB':'Europe','AND':'Europe','ARM':'Europe','AUT':'Europe','AZE':'Europe',
    'BEL':'Europe','BIH':'Europe','BLR':'Europe','BOH':'Europe','BUL':'Europe',
    'CRO':'Europe','CRT':'Europe','CYP':'Europe','CZE':'Europe','DEN':'Europe',
    'ESP':'Europe','EST':'Europe','EUN':'Europe','FIN':'Europe','FRA':'Europe',
    'FRG':'Europe','GBR':'Europe','GDR':'Europe','GEO':'Europe','GER':'Europe',
    'GRE':'Europe','HUN':'Europe','IRL':'Europe','ISL':'Europe','ISR':'Europe',
    'ITA':'Europe','KOS':'Europe','LAT':'Europe','LIE':'Europe','LTU':'Europe',
    'LUX':'Europe','MDA':'Europe','MKD':'Europe','MLT':'Europe','MNE':'Europe',
    'MON':'Europe','NED':'Europe','NOR':'Europe','POL':'Europe','POR':'Europe',
    'ROU':'Europe','RUS':'Europe','SAA':'Europe','SCG':'Europe','SMR':'Europe',
    'SRB':'Europe','SUI':'Europe','SVK':'Europe','SWE':'Europe','SWZ':'Europe',
    'TCH':'Europe','TUR':'Europe','UKR':'Europe','URS':'Europe','YUG':'Europe',
    # Americas
    'ANT':'Americas','ARG':'Americas','ARU':'Americas','BAH':'Americas',
    'BAR':'Americas','BER':'Americas','BOL':'Americas','BRA':'Americas',
    'CAN':'Americas','CAY':'Americas','CHI':'Americas','COL':'Americas',
    'CRC':'Americas','CUB':'Americas','DMA':'Americas','DOM':'Americas',
    'ECU':'Americas','ESA':'Americas','GRN':'Americas','GUA':'Americas',
    'GUY':'Americas','HAI':'Americas','HON':'Americas','ISV':'Americas',
    'IVB':'Americas','JAM':'Americas','LCA':'Americas','MEX':'Americas',
    'NCA':'Americas','NFL':'Americas','PAN':'Americas','PAR':'Americas',
    'PER':'Americas','PUR':'Americas','SKN':'Americas','SUR':'Americas',
    'TTO':'Americas','URU':'Americas','USA':'Americas','VEN':'Americas',
    'VIN':'Americas','WIF':'Americas',
    # Asia
    'AFG':'Asia','ASA':'Asia','BAN':'Asia','BHU':'Asia','BRN':'Asia',
    'BRU':'Asia','CHN':'Asia','FSM':'Asia','GUM':'Asia','HKG':'Asia',
    'INA':'Asia','IND':'Asia','IRI':'Asia','IRQ':'Asia','JOR':'Asia',
    'JPN':'Asia','KAZ':'Asia','KGZ':'Asia','KIR':'Asia','KOR':'Asia',
    'KSA':'Asia','KUW':'Asia','LAO':'Asia','LBA':'Asia','LIB':'Asia',
    'MDV':'Asia','MGL':'Asia','MHL':'Asia','MYA':'Asia','NEP':'Asia',
    'NBO':'Asia','OMA':'Asia','PAK':'Asia','PHI':'Asia','PLE':'Asia',
    'PLW':'Asia','PRK':'Asia','QAT':'Asia','SGP':'Asia','SIN':'Asia',
    'SRI':'Asia','SYR':'Asia','THA':'Asia','TJK':'Asia','TKM':'Asia',
    'TLS':'Asia','TPE':'Asia','UAE':'Asia','UAR':'Asia','UZB':'Asia',
    'VIE':'Asia','VNM':'Asia','YAR':'Asia','YEM':'Asia','YMD':'Asia',
    # Africa
    'AGO':'Africa','ALG':'Africa','ANG':'Africa','BDI':'Africa','BEN':'Africa',
    'BOT':'Africa','BUR':'Africa','CAF':'Africa','CAM':'Africa','CGO':'Africa',
    'CHA':'Africa','CIV':'Africa','CMR':'Africa','COD':'Africa','COM':'Africa',
    'CPV':'Africa','DJI':'Africa','EGY':'Africa','ERI':'Africa','ETH':'Africa',
    'GAB':'Africa','GAM':'Africa','GBS':'Africa','GEQ':'Africa','GHA':'Africa',
    'GUI':'Africa','KEN':'Africa','LBR':'Africa','LES':'Africa','MAD':'Africa',
    'MAR':'Africa','MAW':'Africa','MLI':'Africa','MOZ':'Africa','MRI':'Africa',
    'MTN':'Africa','NAM':'Africa','NGR':'Africa','NIG':'Africa','RHO':'Africa',
    'RSA':'Africa','RWA':'Africa','SEN':'Africa','SEY':'Africa','SLE':'Africa',
    'SOL':'Africa','SOM':'Africa','SSD':'Africa','STP':'Africa','SUD':'Africa',
    'SWZ':'Africa','TAN':'Africa','TOG':'Africa','TUN':'Africa','UGA':'Africa',
    'ZAM':'Africa','ZIM':'Africa',
    # Oceania
    'ANZ':'Oceania','AUS':'Oceania','COK':'Oceania','FIJ':'Oceania',
    'NRU':'Oceania','NZL':'Oceania','PNG':'Oceania','SAM':'Oceania',
    'SOL':'Oceania','TGA':'Oceania','TUV':'Oceania','VAN':'Oceania',
    # Other / Special
    'IOA':'Other','ROT':'Other','UNK':'Other',
}

# ── Clean noc_regions ─────────────────────────────────────────────────────
noc = noc_raw.copy()
noc.columns = noc.columns.str.strip()
noc['region'] = noc['region'].str.strip()

# Fill nulls with NOC code itself (fallback)
noc['region'] = noc['region'].fillna(noc['NOC'])

# Add continent
noc['continent'] = noc['NOC'].map(CONTINENT_MAP).fillna('Other')

# Keep clean notes column
noc['notes'] = noc['notes'].fillna('')

print(f'\nCleaned noc_regions shape: {noc.shape}')
print(f'Continent breakdown:\n{noc["continent"].value_counts()}')
print(f'\nSample:')
print(noc.head(10).to_string())


Cleaned noc_regions shape: (230, 4)
Continent breakdown:
continent
Europe      59
Africa      55
Asia        54
Americas    42
Oceania     12
Other        8
Name: count, dtype: int64

Sample:
   NOC       region                 notes continent
0  AFG  Afghanistan                            Asia
1  AHO      Curacao  Netherlands Antilles     Other
2  ALB      Albania                          Europe
3  ALG      Algeria                          Africa
4  AND      Andorra                          Europe
5  ANG       Angola                          Africa
6  ANT      Antigua   Antigua and Barbuda  Americas
7  ANZ    Australia           Australasia   Oceania
8  ARG    Argentina                        Americas
9  ARM      Armenia                          Europe


In [4]:
# ── Save processed noc_regions ────────────────────────────────────────────
noc.to_csv(PROCESSED / 'noc_regions_clean.csv', index=False)
print('Saved: data/processed/noc_regions_clean.csv')

Saved: data/processed/noc_regions_clean.csv


## 2. Enrich Key Output Tables

In [5]:
# Helper: merge noc lookup into any table that has a 'NOC' column
def enrich_with_region(df: pd.DataFrame, noc_lookup: pd.DataFrame) -> pd.DataFrame:
    """
    Merges region and continent columns into df using NOC as key.

    Args:
        df (pd.DataFrame): Table with a 'NOC' column.
        noc_lookup (pd.DataFrame): Cleaned noc_regions with NOC, region, continent.

    Returns:
        pd.DataFrame: Enriched table with region and continent columns inserted after NOC.
    """
    lookup = noc_lookup[['NOC', 'region', 'continent']].drop_duplicates('NOC')
    merged = df.merge(lookup, on='NOC', how='left')
    # Move region/continent to appear right after NOC
    cols = list(merged.columns)
    noc_idx = cols.index('NOC')
    for col in ['continent', 'region']:  # insert in reverse order
        if col in cols:
            cols.remove(col)
            cols.insert(noc_idx + 1, col)
    return merged[cols]


lookup = noc[['NOC', 'region', 'continent']].drop_duplicates('NOC')

# ── 1. All-time medal table ───────────────────────────────────────────────
alltime = pd.read_csv(TABLES / 'noc_alltime_medal_table.csv')
alltime_rich = enrich_with_region(alltime, noc)
alltime_rich.to_csv(PROCESSED / 'noc_alltime_medal_table_enriched.csv', index=False)
print('All-time medal table enriched:', alltime_rich.shape)
print(alltime_rich.head(5).to_string())

# ── 2. Modern medal table (1984–2016) ────────────────────────────────────
modern = pd.read_csv(TABLES / 'noc_modern_medal_table.csv')
modern_rich = enrich_with_region(modern, noc)
modern_rich.to_csv(PROCESSED / 'noc_modern_medal_table_enriched.csv', index=False)
print('\nModern medal table enriched:', modern_rich.shape)

# ── 3. LA 2028 forecast ───────────────────────────────────────────────────
forecast = pd.read_csv(TABLES / 'forecast_la2028_full.csv')
forecast_rich = enrich_with_region(forecast, noc)
forecast_rich.to_csv(PROCESSED / 'forecast_la2028_enriched.csv', index=False)
print('\nForecast table enriched:', forecast_rich.shape)
print(forecast_rich.head(5).to_string())

All-time medal table enriched: (147, 8)
   NOC   region continent  Bronze  Gold  Silver  Total  Rank
0  USA      USA  Americas    1197  2472    1333   5002     1
1  URS   Russia    Europe     596   832     635   2063     2
2  GBR       UK    Europe     620   636     729   1985     3
3  GER  Germany    Europe     649   592     538   1779     4
4  ITA    Italy    Europe     454   518     474   1446     5

Modern medal table enriched: (127, 8)

Forecast table enriched: (207, 10)
   NOC   region continent  Medals_2016  Pred_2028_Ensemble  Pred_2028_Low  Pred_2028_High  Is_Host  Rank_2028  Delta_vs_2016
0  USA      USA  Americas          264               277.1          273.6           280.6        1          1           13.1
1  GER  Germany    Europe          159               146.7          143.2           150.3        0          2          -12.3
2  GBR       UK    Europe          145               121.4          117.9           125.0        0          3          -23.6
3  FRA   France    

In [6]:
# ── 4. Emerging nations ────────────────────────────────────────────────────
emerging = pd.read_csv(TABLES / 'noc_emerging_nations.csv')
emerging_rich = enrich_with_region(emerging, noc)
emerging_rich.to_csv(PROCESSED / 'noc_emerging_nations_enriched.csv', index=False)
print('Emerging nations enriched:', emerging_rich.shape)

# ── 5. Medal predictions top-line ────────────────────────────────────────
preds = pd.read_csv(TABLES / 'forecast_la2028_medal_predictions.csv')
if 'NOC' in preds.columns:
    preds_rich = enrich_with_region(preds, noc)
    preds_rich.to_csv(PROCESSED / 'forecast_medal_predictions_enriched.csv', index=False)
    print('Medal predictions enriched:', preds_rich.shape)
else:
    print('Predictions table columns:', preds.columns.tolist())

# ── 6. NOC modern medal rate ──────────────────────────────────────────────
rate = pd.read_csv(TABLES / 'noc_modern_medalRate.csv')
if 'NOC' in rate.columns:
    rate_rich = enrich_with_region(rate, noc)
    rate_rich.to_csv(PROCESSED / 'noc_modern_medalRate_enriched.csv', index=False)
    print('Medal rate enriched:', rate_rich.shape)
else:
    print('Medal rate columns:', rate.columns.tolist())

Emerging nations enriched: (34, 5)
Medal predictions enriched: (207, 9)
Medal rate enriched: (141, 7)


## 3. Continental Aggregate Analysis

In [7]:
# ── Load athlete events for continental time-series ─────────────────────
df = pd.read_csv(RAW / 'athlete_events.csv')
summer = df[df['Season'] == 'Summer'].copy()

# Merge continent
summer = summer.merge(lookup, on='NOC', how='left')
summer['continent'] = summer['continent'].fillna('Other')

# Filter post-1948 modern era
modern_games = summer[summer['Year'] >= 1948].copy()
print(f'Modern era records: {len(modern_games):,}')

# Medals only
medals_only = modern_games.dropna(subset=['Medal']).copy()
print(f'Modern era medal records: {len(medals_only):,}')

Modern era records: 186,069
Modern era medal records: 26,187


In [8]:
# ── Chart 1: Medal share by continent over time (stacked area) ─────────
cont_time = (
    medals_only
    .groupby(['Year', 'continent'])
    .size()
    .reset_index(name='medals')
)

# Compute percentage share per year
cont_time['total_year'] = cont_time.groupby('Year')['medals'].transform('sum')
cont_time['share_pct']  = (cont_time['medals'] / cont_time['total_year'] * 100).round(1)

fig1 = px.area(
    cont_time,
    x='Year', y='share_pct', color='continent',
    color_discrete_map=CONTINENT_COLORS,
    title='Olympic Medal Share by Continent (1948–2016)',
    labels={'share_pct': 'Medal Share (%)', 'Year': 'Olympic Year', 'continent': 'Continent'},
    template='plotly_white',
)
fig1.update_layout(**PLOTLY_LAYOUT)
fig1.write_html(FIGURES / 'noc_area_medalShare_byContinent.html')
print('Chart 1 saved: noc_area_medalShare_byContinent.html')
fig1.show()

Chart 1 saved: noc_area_medalShare_byContinent.html


In [9]:
# ── Chart 2: Total medals by continent — all-time bar ──────────────────
cont_total = (
    medals_only
    .groupby(['continent', 'Medal'])
    .size()
    .reset_index(name='count')
)

medal_colors = {'Gold': OLYMPIC_YELLOW, 'Silver': '#C0C0C0', 'Bronze': '#CD7F32'}

cont_order = (
    cont_total.groupby('continent')['count'].sum()
    .sort_values(ascending=False)
    .index.tolist()
)

fig2 = px.bar(
    cont_total,
    x='continent', y='count', color='Medal',
    color_discrete_map=medal_colors,
    category_orders={'continent': cont_order, 'Medal': ['Gold', 'Silver', 'Bronze']},
    title='All-Time Olympic Medals by Continent (1948–2016)',
    labels={'count': 'Total Medals', 'continent': 'Continent'},
    template='plotly_white',
    barmode='stack',
)
fig2.update_layout(**PLOTLY_LAYOUT)
fig2.write_html(FIGURES / 'noc_bar_totalMedals_byContinent.html')
print('Chart 2 saved: noc_bar_totalMedals_byContinent.html')
fig2.show()

Chart 2 saved: noc_bar_totalMedals_byContinent.html


In [10]:
# ── Chart 3: LA 2028 forecast medals by continent ─────────────────────
forecast_rich = pd.read_csv(PROCESSED / 'forecast_la2028_enriched.csv')

cont_forecast = (
    forecast_rich
    .groupby('continent')['Pred_2028_Ensemble']
    .sum()
    .reset_index()
    .rename(columns={'Pred_2028_Ensemble': 'Predicted_Medals'})
    .sort_values('Predicted_Medals', ascending=False)
)
cont_forecast['Predicted_Medals'] = cont_forecast['Predicted_Medals'].round(1)

fig3 = px.bar(
    cont_forecast,
    x='continent', y='Predicted_Medals',
    color='continent',
    color_discrete_map=CONTINENT_COLORS,
    title='LA 2028 Predicted Medal Count by Continent',
    labels={'Predicted_Medals': 'Predicted Medals', 'continent': 'Continent'},
    template='plotly_white',
    text='Predicted_Medals',
)
fig3.update_traces(texttemplate='%{text:.0f}', textposition='outside')
fig3.update_layout(**PLOTLY_LAYOUT)
fig3.write_html(FIGURES / 'noc_bar_forecast2028_byContinent.html')
print('Chart 3 saved: noc_bar_forecast2028_byContinent.html')
print(cont_forecast.to_string())
fig3.show()

Chart 3 saved: noc_bar_forecast2028_byContinent.html
  continent  Predicted_Medals
3    Europe             982.4
1  Americas             547.5
2      Asia             285.9
4   Oceania             133.0
0    Africa              80.3
5     Other              11.3


In [11]:
# ── Chart 4: Continent medal share treemap (modern era) ───────────────
cont_noc_medals = (
    medals_only
    .groupby(['continent', 'region', 'NOC'])
    .size()
    .reset_index(name='medals')
    .sort_values('medals', ascending=False)
)
# Show top 60 NOCs by medal count for readability
top60 = cont_noc_medals.head(60)

fig4 = px.treemap(
    top60,
    path=['continent', 'region'],
    values='medals',
    color='continent',
    color_discrete_map=CONTINENT_COLORS,
    title='Olympic Medal Distribution — Continent & Country (1948–2016, Top 60)',
    template='plotly_white',
)
fig4.update_layout(**PLOTLY_LAYOUT)
fig4.write_html(FIGURES / 'noc_treemap_medals_continent_country.html')
print('Chart 4 saved: noc_treemap_medals_continent_country.html')
fig4.show()

Chart 4 saved: noc_treemap_medals_continent_country.html


## 4. Continental Summary Tables

In [12]:
# ── Table 1: All-time medal summary by continent ──────────────────────
cont_summary = (
    medals_only
    .groupby('continent')
    .agg(
        Total_Medals=('Medal', 'count'),
        Gold=('Medal', lambda x: (x == 'Gold').sum()),
        Silver=('Medal', lambda x: (x == 'Silver').sum()),
        Bronze=('Medal', lambda x: (x == 'Bronze').sum()),
        Unique_NOCs=('NOC', 'nunique'),
        Games_Represented=('Year', 'nunique'),
    )
    .reset_index()
    .sort_values('Total_Medals', ascending=False)
)
cont_summary['Medal_Share_Pct'] = (cont_summary['Total_Medals'] / cont_summary['Total_Medals'].sum() * 100).round(1)

cont_summary.to_csv(PROCESSED / 'continental_medal_summary.csv', index=False)
print('Continental medal summary:')
print(cont_summary.to_string())

Continental medal summary:
  continent  Total_Medals  Gold  Silver  Bronze  Unique_NOCs  Games_Represented  Medal_Share_Pct
3    Europe         15512  4796    5230    5486           49                 18             59.2
1  Americas          5665  2366    1657    1642           28                 18             21.6
2      Asia          2940   933     989    1018           30                 18             11.2
4   Oceania          1495   430     489     576            4                 18              5.7
0    Africa           514   157     151     206           27                 17              2.0
5     Other            61     8      26      27            5                  9              0.2


In [13]:
# ── Table 2: LA 2028 forecast by continent ───────────────────────────
cont_forecast_full = (
    forecast_rich
    .groupby('continent')
    .agg(
        Predicted_2028=('Pred_2028_Ensemble', 'sum'),
        Pred_Low=('Pred_2028_Low', 'sum'),
        Pred_High=('Pred_2028_High', 'sum'),
        Medals_2016=('Medals_2016', 'sum'),
        NOC_Count=('NOC', 'count'),
    )
    .reset_index()
    .sort_values('Predicted_2028', ascending=False)
)
cont_forecast_full['Delta_vs_2016'] = (cont_forecast_full['Predicted_2028'] - cont_forecast_full['Medals_2016']).round(1)
cont_forecast_full[['Predicted_2028', 'Pred_Low', 'Pred_High', 'Medals_2016']] = (
    cont_forecast_full[['Predicted_2028', 'Pred_Low', 'Pred_High', 'Medals_2016']].round(1)
)

cont_forecast_full.to_csv(PROCESSED / 'continental_forecast_2028.csv', index=False)
print('Continental 2028 forecast:')
print(cont_forecast_full.to_string())

Continental 2028 forecast:
  continent  Predicted_2028  Pred_Low  Pred_High  Medals_2016  NOC_Count  Delta_vs_2016
3    Europe           982.4     857.6     1155.4         1053         49          -70.6
1  Americas           547.5     499.3      689.2          472         40           75.5
2      Asia           285.9     230.8      452.2          277         47            8.9
4   Oceania           133.0     121.8      171.9          131         11            2.0
0    Africa            80.3      47.7      271.9           75         54            5.3
5     Other            11.3       3.6       29.1           14          5           -2.7


## 5. Summary

In [14]:
print('=' * 60)
print('NOTEBOOK 07 — COMPLETE')
print('=' * 60)
print()
print('FILES SAVED:')
print('  data/processed/noc_regions_clean.csv')
print('  data/processed/noc_alltime_medal_table_enriched.csv')
print('  data/processed/noc_modern_medal_table_enriched.csv')
print('  data/processed/forecast_la2028_enriched.csv')
print('  data/processed/noc_emerging_nations_enriched.csv')
print('  data/processed/continental_medal_summary.csv')
print('  data/processed/continental_forecast_2028.csv')
print()
print('CHARTS SAVED (outputs/figures/):')
print('  noc_area_medalShare_byContinent.html')
print('  noc_bar_totalMedals_byContinent.html')
print('  noc_bar_forecast2028_byContinent.html')
print('  noc_treemap_medals_continent_country.html')
print()
print('KEY FINDINGS:')
print(f'  Europe dominates all-time Olympic medals')
print(f'  Americas driven by USA host advantage in 2028')
print(f'  Asia rising — China + Japan growth trend')
print()
print('NEXT: Milestone 9 — PowerPoint Consulting Deck')

NOTEBOOK 07 — COMPLETE

FILES SAVED:
  data/processed/noc_regions_clean.csv
  data/processed/noc_alltime_medal_table_enriched.csv
  data/processed/noc_modern_medal_table_enriched.csv
  data/processed/forecast_la2028_enriched.csv
  data/processed/noc_emerging_nations_enriched.csv
  data/processed/continental_medal_summary.csv
  data/processed/continental_forecast_2028.csv

CHARTS SAVED (outputs/figures/):
  noc_area_medalShare_byContinent.html
  noc_bar_totalMedals_byContinent.html
  noc_bar_forecast2028_byContinent.html
  noc_treemap_medals_continent_country.html

KEY FINDINGS:
  Europe dominates all-time Olympic medals
  Americas driven by USA host advantage in 2028
  Asia rising — China + Japan growth trend

NEXT: Milestone 9 — PowerPoint Consulting Deck
